# Project: Facial Recognition  
### Soren Patel

## Imports and Setup

In [1]:
import os
import glob
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from facenet_pytorch import MTCNN, InceptionResnetV1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize MTCNN for single-face extraction and InceptionResnetV1 for embeddings
mtcnn = MTCNN(image_size=160, margin=0, keep_all=False, device=device).eval()
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

Using device: cpu


  0%|          | 0.00/107M [00:00<?, ?B/s]

In [2]:
def get_embedding(img_path):
    img = Image.open(img_path).convert('RGB')
    face = mtcnn(img)
    if face is None:
        return None
    face = face.unsqueeze(0).to(device)
    with torch.no_grad():
        emb = resnet(face)
        emb = F.normalize(emb, p=2, dim=1)  # L2 normalize for cosine similarity
    return emb.squeeze(0).cpu()

Define image path and class alphas

In [3]:
# Defined class roster alphas
class_alphas = {'273126', '273480', '274260', '275262', '275352', '275580', '276516', '276546', '284932', '285094'}

gallery_paths = sorted(glob.glob('imgs/*.jpg'))

# Precompute embeddings and sort into the 3 galleries
galleries = {
    'class': {'embs': [], 'alphas': [], 'paths': []},
    '1C': {'embs': [], 'alphas': [], 'paths': []},
    'all': {'embs': [], 'alphas': [], 'paths': []}
}

for path in gallery_paths:
    alpha = os.path.splitext(os.path.basename(path))[0]
    emb = get_embedding(path)
    
    if emb is not None:
        # All midshipmen
        galleries['all']['embs'].append(emb)
        galleries['all']['alphas'].append(alpha)
        galleries['all']['paths'].append(path)
        
        # 1/C midshipmen
        if alpha.startswith('27'):
            galleries['1C']['embs'].append(emb)
            galleries['1C']['alphas'].append(alpha)
            galleries['1C']['paths'].append(path)
            
        # Class midshipmen
        if alpha in class_alphas:
            galleries['class']['embs'].append(emb)
            galleries['class']['alphas'].append(alpha)
            galleries['class']['paths'].append(path)

# Stack embeddings into tensors for fast matrix multiplication
for g in galleries.values():
    if g['embs']:
        g['embs'] = torch.stack(g['embs'])

print(f"Loaded faces — Class: {len(galleries['class']['alphas'])}, 1/C: {len(galleries['1C']['alphas'])}, All: {len(galleries['all']['alphas'])}")

KeyboardInterrupt: 

In [ ]:
def recognize_face(probe_path, gallery_name='all'):
    probe_emb = get_embedding(probe_path)
    if probe_emb is None:
        print(f"Could not detect a face in {probe_path}")
        return

    g = galleries[gallery_name]
    k = min(5, len(g['alphas']))
    
    # Cosine similarities via dot product
    sims = torch.mv(g['embs'], probe_emb)
    top_scores, top_idx = torch.topk(sims, k=k)

    # Plot probe side-by-side with top matches
    fig, axes = plt.subplots(1, k + 1, figsize=(15, 3))
    
    axes[0].imshow(Image.open(probe_path).convert('RGB'))
    axes[0].set_title(f"Probe\n{os.path.basename(probe_path)}")
    axes[0].axis('off')

    for i in range(k):
        match_idx = top_idx[i].item()
        match_img = Image.open(g['paths'][match_idx]).convert('RGB')
        
        axes[i + 1].imshow(match_img)
        axes[i + 1].set_title(f"#{i+1}: {g['alphas'][match_idx]}\nSim: {top_scores[i]:.2f}")
        axes[i + 1].axis('off')

    plt.suptitle(f"Matches in Gallery: {gallery_name.upper()}", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Put the path to your probe image here
probe_img_path = "test_probe.jpg"

for g_name in ['class', '1C', 'all']:
    recognize_face(probe_img_path, gallery_name=g_name)